# Pratica — Lezione 9: completa il `text_to_speech` dell'assistente vocale

Nel progetto reale di IBM `chatapp-with-voice-and-openai-outline`, il
file `worker.py` ha tre funzioni: `speech_to_text`,
`openai_process_message` e `text_to_speech`. Le prime due sono
complete nel materiale fonte — ma `text_to_speech` è rimasta con un
`base_url = '...'` segnaposto, senza arrivare a collegarsi al servizio di
sintesi vocale (Watson TTS, a pagamento).

In questa pratica risolverai tu proprio questo vuoto, usando `gTTS`
(Google Text-to-Speech), l'alternativa 100% gratuita che usiamo nel
notebook di lezione. Il resto della pipeline (STT con Whisper, l'LLM con
SmolLM2) è già risolto in modo che tu ti concentri sul pezzo mancante.

In [ ]:
# Installiamo le librerie necessarie.
!pip install -q gTTS transformers accelerate

## Esercizio: completa `text_to_speech`

Qui sotto trovi l'impalcatura completa della pipeline. Le funzioni
`speech_to_text` e `llm_process_message` sono già risolte (proprio come nel
`worker.py` reale). Il tuo compito è completare `text_to_speech` seguendo le
istruzioni nei commenti `# TODO`.

Suggerimento: `gTTS` si usa così — `gTTS(text=texto, lang=idioma)` crea l'oggetto
audio, e `.save(ruta)` lo salva come file `.mp3`.

In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch
from gtts import gTTS
from IPython.display import Audio

# --- Già risolto: Speech-to-Text ---
stt_pipe = pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en")

def speech_to_text(audio_path):
    resultado = stt_pipe(audio_path)
    return resultado["text"].strip()


# --- Già risolto: l'LLM ---
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

def llm_process_message(user_message, system_prompt="Sei un assistente breve e cordiale."):
    mensajes = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    entrada = tokenizer.apply_chat_template(
        mensajes, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    with torch.inference_mode():
        salida = llm_model.generate(entrada, max_new_tokens=80, do_sample=True, temperature=0.7)
    texto_generado = tokenizer.decode(salida[0][entrada.shape[1]:], skip_special_tokens=True)
    return texto_generado.strip()


# --- TODO: completa questa funzione ---
# Questa è la funzione che, nel worker.py reale del materiale di IBM, è rimasta
# con un segnaposto "base_url = '...'" non terminato.
def text_to_speech(text, salida_path="respuesta.mp3", idioma="es"):
    """Deve convertire `text` in un file audio salvato in `salida_path`
    e restituire `salida_path`."""
    # TODO: crea un oggetto gTTS con il testo e la lingua indicati
    voz = None  # <-- sostituisci questo

    # TODO: salva l'audio generato in salida_path usando voz.save(...)

    # TODO: restituisci salida_path
    pass

## Testiamo la tua soluzione

Genera un audio di esempio, trascrivilo, passalo attraverso l'LLM e usa il tuo
`text_to_speech` per ascoltare la risposta. Se `text_to_speech` è stata
completata correttamente, la cella qui sotto dovrebbe riprodurre un audio con la
risposta dell'assistente.

In [ ]:
gTTS(text="What is the capital of France?", lang="en").save("audio_usuario.mp3")

texto_transcrito = speech_to_text("audio_usuario.mp3")
print("Testo trascritto:", texto_transcrito)

respuesta = llm_process_message(texto_transcrito)
print("Risposta dell'LLM:", respuesta)

ruta_audio_respuesta = text_to_speech(respuesta, idioma="en")
Audio(ruta_audio_respuesta)

<details>
<summary>💡 Vedi soluzione</summary>

```python
def text_to_speech(text, salida_path="respuesta.mp3", idioma="es"):
    """Converte il testo in un file audio e restituisce il percorso del file."""
    # Creiamo l'oggetto gTTS con il testo e la lingua richiesti.
    voz = gTTS(text=text, lang=idioma)

    # Salviamo l'audio generato nel percorso indicato.
    voz.save(salida_path)

    # Restituiamo il percorso del file affinché chi chiama la funzione
    # possa riprodurlo o inviarlo di nuovo al frontend.
    return salida_path
```

Questo è, in sostanza, esattamente ciò che fa la versione a pagamento con Watson TTS:
riceve un testo, genera un file audio, e restituisce un riferimento a quell'
audio (qui un percorso locale; nel `worker.py` reale, i byte dell'audio
restituiti dall'API di Watson).
</details>

## Per continuare

- Notebook di lezione: `notebooks_clase/clase09_asistente_voz_conversacional.ipynb`
- Capitolo del manuale: `manual/13_asistente_voz_conversacional.md`

Sfida extra (opzionale): cambia il `system_prompt` di `llm_process_message`
affinché l'assistente risponda come un terapeuta, un meccanico, un
narratore o un professore — proprio come suggeriscono gli esercizi di "prompt
engineering" del materiale originale di IBM — e osserva come cambia il tono
della risposta (e, di conseguenza, dell'audio generato).